In [25]:
!pip install numpy pandas nltk textblob wordcloud seaborn matplotlib cufflinks plotly

In [34]:
!pip install numpy pandas nltk textblob wordcloud seaborn matplotlib plotly cufflinks scikit-learn vaderSentiment transformers torch spacy gensim


In [38]:
import numpy as np
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import re
from textblob import TextBlob
from wordcloud import WordCloud
import seaborn as sns
import matplotlib.pyplot as plt
import cufflinks as cf

# Fix Matplotlib Magic Command
%matplotlib inline

# Fix Plotly Import
from plotly.offline import init_notebook_mode, iplot
init_notebook_mode(connected=True)
cf.go_offline()

# Correct Import for Subplots
from plotly.subplots import make_subplots

# Correct Import for Warnings
import warnings
warnings.filterwarnings("ignore")
warnings.warn("This will not show")

# Fix Pandas Display Option
pd.set_option('display.max_columns', None)

In [40]:
#importing dataset

df=pd.read_csv("amazon.csv")
df

,Unnamed: 0,reviewerName,overall,reviewText,reviewTime,day_diff,helpful_yes,helpful_no,total_vote,score_pos_neg_diff,score_average_rating,wilson_lower_bound
0,0,NaN,4,No issues.,23-07-2014,138,0,0,0,0,0.0,0.0
1,1,0mie,5,"Purchased this for my device, it worked as adv...",25-10-2013,409,0,0,0,0,0.0,0.0
2,2,1K3,4,it works as expected. I should have sprung for...,23-12-2012,715,0,0,0,0,0.0,0.0
3,3,1m2,5,This think has worked out great.Had a diff. br...,21-11-2013,382,0,0,0,0,0.0,0.0
4,4,2&amp;1/2Men,5,"Bought it with Retail Packaging, arrived legit...",13-07-2013,513,0,0,0,0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
4910,4910,"ZM ""J""",1,I bought this Sandisk 16GB Class 10 to use wit...,23-07-2013,503,0,0,0,0,0.0,0.0
4911,4911,Zo,5,Used this for extending the capabilities of my...,22-08-2013,473,0,0,0,0,0.0,0.0
4912,4912,Z S Liske,5,Great card that is very fast and reliable. It ...,31-03-2014,252,0,0,0,0,0.0,0.0
4913,4913,Z Taylor,5,Good amount of space for the stuff I want to d...,16-09-2013,448,0,0,0,0,0.0,0.0


In [49]:
import pandas as pd
import numpy as np

def missing_value_analysis(df):
    na_columns_ = [col for col in df.columns if df[col].isnull().sum() > 0]
    n_miss = df[na_columns_].isnull().sum().sort_values(ascending=True)
    ratio_ = (df[na_columns_].isnull().sum() / df.shape[0] * 100).sort_values(ascending=True)
    missing_df = pd.concat([n_miss, np.round(ratio_, 2)], axis=1, keys=['Missing Values', 'Ratio'])
    return missing_df

def check_dataframe(df, head=5, tail=5):
    print("Shape".center(82, '~'))
    print('Rows: {}'.format(df.shape[0]))
    print('Columns: {}'.format(df.shape[1]))
    print("TYPES".center(82, '~'))
    print(df.dtypes)
    print("".center(82, '~'))
    print(missing_value_analysis(df))
    print('DUPLICATE VALUES'.center(83, '~'))
    print(df.duplicated().sum())  # Corrected from df.duplicates() to df.duplicated()
    print('QUANTILES'.center(82, '~'))
    
    # Calculate quantiles only for numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        print(df[numeric_cols].quantile([0, 0.05, 0.50, 0.95, 0.99, 1]).T)
    else:
        print("No numeric columns to calculate quantiles.")
# Example usage
# df = pd.read_csv('your_data.csv')  # Load your DataFrame here
check_dataframe(df)

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~Shape~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Rows: 4915
Columns: 11
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~TYPES~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
reviewerName             object
overall                   int64
reviewText               object
reviewTime               object
day_diff                  int64
helpful_yes               int64
helpful_no                int64
total_vote                int64
score_pos_neg_diff        int64
score_average_rating    float64
wilson_lower_bound      float64
dtype: object
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
              Missing Values  Ratio
reviewerName               1   0.02
reviewText                 1   0.02
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~DUPLICATE VALUES~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
0
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~QUANTILES~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
                       0.00  0.05   0.50        0.95       0.99         1.00
overall 

In [45]:
import pandas as pd

def check_class(dataframe):
    nunique_df = pd.DataFrame({
        'Variable': dataframe.columns,
        'Classes': [dataframe[i].nunique()
                    for i in dataframe.columns]})
    
    nunique_df = nunique_df.sort_values('Classes', ascending=False)  # Sort by count of unique values
    nunique_df = nunique_df.reset_index(drop=True)
    
    return nunique_df

# Example usage
check_class(df)


,Variable,Classes
0,Unnamed: 0,4915
1,reviewText,4912
2,reviewerName,4594
3,reviewTime,690
4,day_diff,690
5,wilson_lower_bound,40
6,score_average_rating,28
7,score_pos_neg_diff,27
8,total_vote,26
9,helpful_yes,23


In [66]:
constraints = ['#B34D22', '#EBE00C', '#1FEB0C', '#0C92EB', '#EB0CD5']

def categorical_variable_summary(df, column_name):
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=('Countplot', 'Percentage'),
                        specs=[[{"type": "xy"}, {'type': 'domain'}]])
    
    # Countplot
    fig.add_trace(go.Bar(
        y=df[column_name].value_counts().values.tolist(),
        x=[str(i) for i in df[column_name].value_counts().index],
        text=df[column_name].value_counts().values.tolist(),
        textfont=dict(size=12),  # Adjust text font size if needed
        name= column_name,
        textposition='auto',
        showlegend=False,
        marker=dict(color=constraints,
                    line=dict(color='#DBE0EC', width=1))
    ), row=1, col=1)

    # Percentage Pie Chart
    fig.add_trace(go.Pie(
        labels=df[column_name].value_counts().keys(),
        values=df[column_name].value_counts().values,
        textfont=dict(size=18),
        textposition='auto',
        showlegend=False,
        name=column_name,
        marker=dict(colors=constraints)  # Changed 'color' to 'colors'
    ), row=1, col=2)

    fig.update_layout(title={'text': column_name,
                              'y': 0.9,
                              'x': 0.5,
                              'xanchor': 'center',
                              'yanchor': 'top'},
                      template='plotly_white')
    
    # Use plot instead of iplot to display in the notebook
    iplot(fig)

# Example usage
categorical_variable_summary(df, 'overall')

In [48]:
# arrange wilson_lower_bound in ascending order

df=df.sort_values("wilson_lower_bound",ascending=False)
df.drop('Unnamed: 0', inplace=True, axis=1)
df.head()

,reviewerName,overall,reviewText,reviewTime,day_diff,helpful_yes,helpful_no,total_vote,score_pos_neg_diff,score_average_rating,wilson_lower_bound
2031,"Hyoun Kim ""Faluzure""",5,[[ UPDATE - 6/19/2014 ]]So my lovely wife boug...,05-01-2013,702,1952,68,2020,1884,0.966337,0.957544
3449,NLee the Engineer,5,I have tested dozens of SDHC and micro-SDHC ca...,26-09-2012,803,1428,77,1505,1351,0.948837,0.936519
4212,SkincareCEO,1,NOTE: please read the last update (scroll to ...,08-05-2013,579,1568,126,1694,1442,0.925620,0.912139
317,"Amazon Customer ""Kelly""",1,"If your card gets hot enough to be painful, it...",09-02-2012,1033,422,73,495,349,0.852525,0.818577
4672,Twister,5,Sandisk announcement of the first 128GB micro ...,03-07-2014,158,45,4,49,41,0.918367,0.808109


In [49]:
df.reviewText.head()

2031    [[ UPDATE - 6/19/2014 ]]So my lovely wife boug...
3449    I have tested dozens of SDHC and micro-SDHC ca...
4212    NOTE:  please read the last update (scroll to ...
317     If your card gets hot enough to be painful, it...
4672    Sandisk announcement of the first 128GB micro ...
Name: reviewText, dtype: object

In [50]:
review_example =df.reviewText[2031]
review_example

'[[ UPDATE - 6/19/2014 ]]So my lovely wife bought me a Samsung Galaxy Tab 4 for Father\'s Day and I\'ve been loving it ever since.  Just as other with Samsung products, the Galaxy Tab 4 has the ability to add a microSD card to expand the memory on the device.  Since it\'s been over a year, I decided to do some more research to see if SanDisk offered anything new.  As of 6/19/2014, their product lineup for microSD cards from worst to best (performance-wise) are the as follows:SanDiskSanDisk UltraSanDisk Ultra PLUSSanDisk ExtremeSanDisk Extreme PLUSSanDisk Extreme PRONow, the difference between all of these cards are simply the speed in which you can read/write data to the card.  Yes, the published rating of most all these cards (except the SanDisk regular) are Class 10/UHS-I but that\'s just a rating... Actual real world performance does get better with each model, but with faster cards come more expensive prices.  Since Amazon doesn\'t carry the Ultra PLUS model of microSD card, I had 

In [51]:
review_example= re.sub("[^a-zA-Z]", '', review_example)
review_example

'UPDATESomylovelywifeboughtmeaSamsungGalaxyTabforFathersDayandIvebeenlovingiteversinceJustasotherwithSamsungproductstheGalaxyTabhastheabilitytoaddamicroSDcardtoexpandthememoryonthedeviceSinceitsbeenoverayearIdecidedtodosomemoreresearchtoseeifSanDiskofferedanythingnewAsoftheirproductlineupformicroSDcardsfromworsttobestperformancewisearetheasfollowsSanDiskSanDiskUltraSanDiskUltraPLUSSanDiskExtremeSanDiskExtremePLUSSanDiskExtremePRONowthedifferencebetweenallofthesecardsaresimplythespeedinwhichyoucanreadwritedatatothecardYesthepublishedratingofmostallthesecardsexcepttheSanDiskregularareClassUHSIbutthatsjustaratingActualrealworldperformancedoesgetbetterwitheachmodelbutwithfastercardscomemoreexpensivepricesSinceAmazondoesntcarrytheUltraPLUSmodelofmicroSDcardIhadtododirectcomparisonsbetweentheSanDiskUltraExtremeandExtremePLUSAsmentionedinmyearlierreviewIpurchasedtheSanDiskUltraformyGalaxySMyquestionwasdidIwanttopayovermoreforacardthatisfasterthantheoneIalreadyownedOrIcouldpayalmostdoubletoget

In [58]:
rt = lambda x: re.sub("[^a-zA-Z]", ' ', str(x))  # Removes non-alphabetic characters
df["reviewText"] = df["reviewText"].map(rt)
df["reviewText"] = df["reviewText"].str.lower()
print(df.head())

                 reviewerName  overall  \
2031     Hyoun Kim "Faluzure"        5   
3449        NLee the Engineer        5   
4212              SkincareCEO        1   
317   Amazon Customer "Kelly"        1   
4672                  Twister        5   

                                             reviewText  reviewTime  day_diff  \
2031     update               so my lovely wife boug...  05-01-2013       702   
3449  i have tested dozens of sdhc and micro sdhc ca...  26-09-2012       803   
4212  note   please read the last update  scroll to ...  08-05-2013       579   
317   if your card gets hot enough to be painful  it...  09-02-2012      1033   
4672  sandisk announcement of the first    gb micro ...  03-07-2014       158   

      helpful_yes  helpful_no  total_vote  score_pos_neg_diff  \
2031         1952          68        2020                1884   
3449         1428          77        1505                1351   
4212         1568         126        1694                1442   


In [62]:
import pandas as pd
import re
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Assuming df["reviewText"] contains text data

# Sentiment Analysis using TextBlob (polarity & subjectivity)
df[['polarity', 'subjectivity']] = df['reviewText'].apply(lambda text: pd.Series(TextBlob(text).sentiment))

# Initialize Sentiment Analyzer
analyzer = SentimentIntensityAnalyzer()

# Sentiment Classification using VADER
for index, row in df['reviewText'].items():  # ✅ Fixed: Replaced iteritems() with items()
    score = analyzer.polarity_scores(row)
    neg, neu, pos = score['neg'], score['neu'], score['pos']

    if neg > pos:
        df.loc[index, 'sentiment'] = "Negative"
    elif pos > neg:
        df.loc[index, 'sentiment'] = "Positive"
    else:
        df.loc[index, 'sentiment'] = "Neutral"

# Print updated DataFrame
print(df.head())


                 reviewerName  overall  \
2031     Hyoun Kim "Faluzure"        5   
3449        NLee the Engineer        5   
4212              SkincareCEO        1   
317   Amazon Customer "Kelly"        1   
4672                  Twister        5   

                                             reviewText  reviewTime  day_diff  \
2031     update               so my lovely wife boug...  05-01-2013       702   
3449  i have tested dozens of sdhc and micro sdhc ca...  26-09-2012       803   
4212  note   please read the last update  scroll to ...  08-05-2013       579   
317   if your card gets hot enough to be painful  it...  09-02-2012      1033   
4672  sandisk announcement of the first    gb micro ...  03-07-2014       158   

      helpful_yes  helpful_no  total_vote  score_pos_neg_diff  \
2031         1952          68        2020                1884   
3449         1428          77        1505                1351   
4212         1568         126        1694                1442   


In [63]:
df[df['sentiment']=='Positive'].sort_values("wilson_lower_bound",ascending=False).head(5)

,reviewerName,overall,reviewText,reviewTime,day_diff,helpful_yes,helpful_no,total_vote,score_pos_neg_diff,score_average_rating,wilson_lower_bound,polarity,subjectivity,sentiment
2031,"Hyoun Kim ""Faluzure""",5,update so my lovely wife boug...,05-01-2013,702,1952,68,2020,1884,0.966337,0.957544,0.163859,0.562259,Positive
3449,NLee the Engineer,5,i have tested dozens of sdhc and micro sdhc ca...,26-09-2012,803,1428,77,1505,1351,0.948837,0.936519,0.103870,0.516435,Positive
4212,SkincareCEO,1,note please read the last update scroll to ...,08-05-2013,579,1568,126,1694,1442,0.925620,0.912139,0.212251,0.505394,Positive
317,"Amazon Customer ""Kelly""",1,if your card gets hot enough to be painful it...,09-02-2012,1033,422,73,495,349,0.852525,0.818577,0.143519,0.494207,Positive
4672,Twister,5,sandisk announcement of the first gb micro ...,03-07-2014,158,45,4,49,41,0.918367,0.808109,0.172332,0.511282,Positive


In [65]:
categorical_variable_summary(df,'sentiment')